Implementation of a basic transformer from start to finish. 
Code is mostly taken from CPSC4770 HW2 part 1

In [3]:
# Keep runtime deps in sync for this notebook environment.
%pip install -q transformers tokenizers

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch import Tensor
from typing import Tuple, List

import random
import math
import os
import time
import json
import numpy as np

# We'll set the random seeds for deterministic results.
SEED = 1

random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.enabled = False
torch.backends.cudnn.deterministic = True

class Placeholder:
    @property
    def DO(self):
        raise NotImplementedError("You haven't yet implemented this part of the assignment yet")

TO = Placeholder()

Note: you may need to restart the kernel to use updated packages.


In [4]:
import torch

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")
print("Pytorch version is: ", torch.__version__)
print("You are using: ", DEVICE)

Pytorch version is:  2.7.0
You are using:  mps


In [5]:
# The first thing we want to do is tokenization

from transformers import GPT2Tokenizer

# Load pre-trained model tokenizer (vocabulary)
# Basically GPT2 knows how to split text into tokens. When we create a tokenizer object, it downloads the vocabulary and the rules for tokenization.
# We can then use this tokenizer to convert text into token IDs that the model can understand.
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')

# this is basically the text we want to tokenize
text = "This adaptation of the enigmatic novel by Liane Moriarty is supremely watchable but flawed."

# we can use tokenizer.encode to convert text to token IDs. I also print the decoded tokens and how the tokenizer
# actually converts the ids back to tokens (when we use convert_ids_to_tokens, we get the tokenizer’s internal token representation)
tokens_ids = tokenizer.encode(text)
for token_id in tokens_ids:
    print(token_id, tokenizer.decode(token_id), tokenizer.convert_ids_to_tokens(token_id))


/Users/timli/miniconda3/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


1212 This This
16711  adaptation Ġadaptation
286  of Ġof
262  the Ġthe
48584  enigmatic Ġenigmatic
5337  novel Ġnovel
416  by Ġby
406  L ĠL
46470 iane iane
32709  Mori ĠMori
25494 arty arty
318  is Ġis
17700  supreme Ġsupreme
306 ly ly
2342  watch Ġwatch
540 able able
475  but Ġbut
19556  flawed Ġflawed
13 . .


In [6]:
# Now we want to implement the embedding layer
#
# So why do we embed?
# The language model can not work directly with words like: ["This", "movie", "is", "great"] or even token IDs like [1212, 345, 6789].
# We need to convert these token IDs into dense vectors that capture the semantic meaning of the tokens. This is what the embedding layer does.

import torch
import torch.nn as nn
from torch import Tensor

class Embedding(nn.Module):
    def __init__(self, vocab_size: int, d_model: int):
        super(Embedding, self).__init__()
        # here we basically say let's say we have 50,000 possible tokens in our vocabulary -> we map each token to a d_model dimensional vector.
        # wte stands for "word token embedding"
        self.wte = nn.Embedding(vocab_size, d_model)

    # this defines what happens when data is passed through the layer
    # let's say we have an input x = torch.tensor([10, 25, 7, 91]) -> here we have a sequence length of 4
    # after passing it through the forward function we get (sequence_length, d_model)
    def forward(self, x: Tensor) -> Tensor:
        return self.wte(x)

# tests
vocab_size = 10
d_model = 16

embedding = Embedding(vocab_size, d_model)
x = torch.tensor([1, 2, 3, 4])
output = embedding(x)
assert output.shape == (4, d_model)


In [7]:
# Now let's look at our positional embedding layers

# The goal of positional embedding is to give the model information about the position of each token in the sequence.
# This is crucial because the transformer architecture does not have any inherent notion of order (unlike RNNs or CNNs).

class PositionalEmbeddings(nn.Module):
    def __init__(self, d_model: int, max_len: int):
        super().__init__()
        # here we create an embedding layer that maps each position (from 0 to max_len-1) to a d_model dimensional vector
        # basically row 0 will be the positional embedding for position 0, etc.
        # if max_len is 512, we will have 512 positional embeddings, each of size d_model
        self.positional_embeddings = nn.Embedding(max_len, d_model)


    def forward(self, x: Tensor) -> Tensor:
        # here we assume x is a tensor of shape (sequence_length,) containing the positions of the tokens in the sequence
        # when we pass x through the positional embedding layer, we get (sequence_length, d_model)
        return self.positional_embeddings(x)



# tests

d_model = 16
max_len = 64
positional_embeddings = PositionalEmbeddings(d_model, max_len)
x = torch.tensor([1, 2, 3, 4])
output = positional_embeddings(x)
assert output.shape == (4, d_model)


# The code above is learned positional embeddings, which means that the model will learn a unique embedding for each position during training.
# However, in the original transformer paper, they used sinusoidal positional embeddings, which are fixed and do not require learning. Let's implement sinusoidal positional embeddings as well.
class PositionalEmbeddings2(nn.Module):
    def __init__(self, d_model: int, max_len: int):
        super().__init__()
        # here we create a matrix of shape (max_len, d_model) where each row corresponds to the positional embedding for that position
        pe = torch.zeros(max_len, d_model)
        # now we create the positions tensor which is [[0], [1], [2], ..., [max_len-1]]
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)

        div_term = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float) * (-math.log(10000.0) / d_model))
        # we fill the even indices of the positional embedding with sine functions and the odd indices with cosine functions
        # this means that for position i, the embedding will have sin(i / 10000^(2j/d_model)) at even indices and cos(i / 10000^(2j/d_model)) at odd indices
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.register_buffer("pe", pe)
    # the input x is a tensor of shape (sequence_length, ) containing the positions of the tokens in the sequence
    def forward(self, x: Tensor) -> Tensor:
        # after passing x through the forward function, we get (sequence_length, d_model) where each row corresponds to the positional embedding for that position
        return self.pe[x]


In [8]:
# now we can combine the token and positional embeddings

class TokenEmbedder(nn.Module):
    def __init__(self, vocab_size: int, d_model: int, max_len: int):
        super(TokenEmbedder, self).__init__()
        self.token_embedding = Embedding(vocab_size, d_model)
        self.positional_embedding = PositionalEmbeddings(d_model, max_len)

    def forward(self, x: Tensor) -> Tensor:
        # add the token embeddings and positional embeddings together
        pos = torch.arange(0, x.shape[1], dtype=torch.long, device=x.device) # shape: [sequence length]
        # here we can just add the token embeddings and positional embeddings together because they have the same shape (sequence_length, d_model)
        return self.token_embedding(x) + self.positional_embedding(pos)

# testing
# test the whole input pipeline

sample_texts = ["This adaptation of the enigmatic novel by Liane Moriarty is supremely watchable but flawed.",
                "The story is a bit of a slow burn, but the performances are top-notch and the ending is worth the wait."]

# encode the text
vocab_size = tokenizer.vocab_size
d_model = 64

# return_tensors="pt" returns pytorch tensors directly. truncation and padding are used to ensure the input length is the same
tokenizer.pad_token = tokenizer.eos_token
token_ids = tokenizer(sample_texts, return_tensors="pt", max_length=64, padding="longest", truncation=True)['input_ids']

token_embedder = TokenEmbedder(vocab_size=tokenizer.vocab_size, d_model=768, max_len=64)

# pass the token_ids to the token_embedder
output = token_embedder(token_ids)

output.shape

torch.Size([2, 26, 768])

Now that we have implemented the input pipeline we can move on to the transformer model architecture.

The first thing we need to do is implement the QKV projections.

In the transformer architecture, we have three matrices: Query (Q), Key (K), and Value (V).
These are linear projections of the input embeddings. The idea is that for each token in the sequence, we want to compute a query vector, a key vector, and a value vector.
The attention mechanism will then use these vectors to compute the attention scores and the weighted sum of the values.


What is the query vector? (Q)
Represents what information you are currently looking for.
For a specific word you’re processing, the Query vector asks, “What other words are relevant to understanding this word?”

What is the key vector? (K)
Key (K): Represents the “label” or identifier for the information available.
Each word in the sequence has a Key vector essentially saying, “Here’s the kind of information I hold.”

What is the value vector? (V)
Value (V): Represents the actual information content.
Each word also has a Value vector, which is the information to be retrieved if that word is deemed relevant.


In [ ]:

import torch
import torch.nn as nn

class QKVProjection(nn.Module):
    def __init__(self, d_model, num_heads, d_k):
        super(QKVProjection, self).__init__()
        assert num_heads * d_k == d_model, "d_model must be equal to num_heads * d_k"
        self.num_heads = num_heads
        self.d_k = d_k

        # we create three linear layers for Q, K, and V.
        # Here we are projecting the input from d_model to num_heads * d_k, which is the total dimension for all heads combined.
        self.q_linear = nn.Linear(d_model, num_heads * d_k)
        self.k_linear = nn.Linear(d_model, num_heads * d_k)
        self.v_linear = nn.Linear(d_model, num_heads * d_k)

    def forward(self, X):
        batch_size, seq_length, d_model = X.shape

        # here we apply the linear layers to get Q, K, V of shape (batch_size, seq_length, num_heads * d_k)
        # we are basically applying the learned linear transformation in self.q_linear to each token embedding which would be like a singular vector of size d_model
        Q = self.q_linear(X)
        K = self.k_linear(X)
        V = self.v_linear(X)

        # now since we need to do multi-head attention, we need to reshape and transpose Q, K, V so that we can do batched attention
        # after the linear layer, we have shape (batch_size, seq_length, num_heads * d_k) -> and we want to get (batch_size, num_heads, seq_length, d_k)

        # We can apply the following transformations:
        # 1. VIEW: split the last dim into (num_heads, d_k) -> (batch, seq, num_heads, d_k).
        # 2. TRANSPOSE(1, 2): swap seq and num_heads -> (batch, num_heads, seq, d_k).
        #    This way each "head" is (seq, d_k), and we can do batched attention:
        #    scores (batch, num_heads, seq, seq) @ V (batch, num_heads, seq, d_k) -> (batch, num_heads, seq, d_k).
        # Target shape for Q, K, V: (batch_size, num_heads, seq_length, d_k).
        Q = Q.view(batch_size, seq_length, self.num_heads, self.d_k).transpose(1, 2)
        K = K.view(batch_size, seq_length, self.num_heads, self.d_k).transpose(1, 2)
        V = V.view(batch_size, seq_length, self.num_heads, self.d_k).transpose(1, 2)

        return Q, K, V

# tests
d_model = 512  # Embedding size
num_heads = 8  # Number of attention heads
d_k = 64  # Dimension per head (num_heads * d_k must be d_model)

batch_size = 2
seq_length = 10

# Random input tensor
X = torch.rand(batch_size, seq_length, d_model)

# Instantiate and apply QKVProjection
qkv_projection = QKVProjection(d_model, num_heads, d_k)
Q, K, V = qkv_projection(X)

print(Q.shape, K.shape, V.shape)

torch.Size([2, 8, 10, 64]) torch.Size([2, 8, 10, 64]) torch.Size([2, 8, 10, 64])


The next part is multi-head self-attention. This allows the model to weigh the importance of different tokens in the input sequence when computing the output for each token. The main parameters of this layer are the projection matrices $W_Q$, $W_K$, $W_V$ and we have H heads. 

So why do we even use multi-headed self-attention?
Imagine trying to understand a complex sentence. You might simultaneously pay attention to its grammatical structure, the semantic meaning of words, and perhaps coreference (which pronouns refer to which nouns). A single attention mechanism might struggle to focus on all these aspects at once. We can use many different heads to keep track of each of these aspects.

In [10]:
import math
import torch.nn.functional as F

# Large negative value for masking (more numerically stable than -inf, especially on MPS/Metal)
MASK_VALUE = -1e9

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, d_k, attn_pdrop=None):

        # this is initializing the multi-headed attention module as a big picture

        # 1. project input X into Q, K, V
        # 2. split into multiple heads
        # 3. compute attention for each head
        # 4. combine the heads back together
        # 5. apply one final linear layer

        super(MultiHeadAttention, self).__init__()

        assert num_heads * d_k == d_model, "d_model must be equal to num_heads * d_k"

        self.num_heads = num_heads
        self.d_k = d_k
        # here attn_pdrop is the dropout probability for the attention weights.
        # We will apply dropout to the attention weights after softmax to prevent overfitting.
        self.attn_pdrop = attn_pdrop

        # Use the improved QKVProjection
        self.qkv_projection = QKVProjection(d_model, num_heads, d_k)

        # Final output projection
        self.out_linear = nn.Linear(d_model, d_model)

    def forward(self, X, causal_mask=False, attention_mask=None):
        '''
        Args:
            X (torch.Tensor): Input tensor of shape (batch_size, seq_length, d_model).
            causal_mask (bool): If True, apply a causal mask to prevent attending to future tokens.
            attention_mask (torch.Tensor, optional): Padding mask of shape (batch_size, seq_length).
                Contains 1 for real tokens and 0 for padding tokens.
        Returns:
            Tuple[torch.Tensor, torch.Tensor]: A tuple containing:
                - attention_weights (torch.Tensor): Attention weights of shape (batch_size, num_heads, seq_length, seq_length).
                - output (torch.Tensor): Output tensor of shape (batch_size, seq_length, d_model).
        '''
        # Generate Q, K, V using the projection module
        Q, K, V = self.qkv_projection(X)
        batch_size, num_heads, seq_length, d_k = Q.shape

        # Compute scaled dot-product attention for the raw_scores:
        # a) Compute the raw attention scores using the dot product between Q and K.
        # b) Scale the scores by dividing by sqrt(d_k).
        raw_scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)

        # Apply causal mask if causal_mask is True
        # Create an upper triangular matrix of ones using torch.triu(..., diagonal=1)
        # and use masked_fill to set future positions to MASK_VALUE BEFORE applying softmax.

        # Why do we use a casual mask? In autoregressive language modeling, we want to prevent the model from attending to future tokens that it shouldn't have access to during training.
        # The causal mask ensures that each token can only attend to itself and previous tokens, not future ones.

        if causal_mask:
            causal = torch.triu(torch.ones(seq_length, seq_length, device=X.device), diagonal=1).bool()
            raw_scores = raw_scores.masked_fill(causal, MASK_VALUE)

        # Apply the padding attention mask (attention_mask) to the scores
        # The attention_mask has shape (batch_size, seq_length) with 1 for real tokens and 0 for padding.
        # You need to:
        #   1. Identify padding positions (where attention_mask == 0)
        #   2. Expand the mask to shape (batch_size, 1, 1, seq_length) so it broadcasts over heads and query positions
        #   3. Use masked_fill to set those positions to MASK_VALUE BEFORE softmax

        # What are padding tokens? In NLP tasks, we will often have sentences with different lengths. Therefore, we need to
        # pad shorter sentences with special padding tokens to ensure that all sentences in a batch have the same length.
        # The attention_mask helps the model ignore these padding tokens during attention computation.
        if attention_mask is not None:
            padding_positions = (attention_mask == 0)
            padding_mask = padding_positions.view(batch_size, 1, 1, seq_length)
            raw_scores = raw_scores.masked_fill(padding_mask, MASK_VALUE)

        # Apply softmax to get attention weights
        attention_weights = F.softmax(raw_scores, dim =-1)

        # Apply dropout to the attention weights only during training

        # Why do we apply dropout to the attention weights?
        # Dropout is a regularization technique that helps prevent overfitting by randomly setting some of the attention weights to zero during training.
        # This encourages the model to not rely too heavily on any single token and promotes better generalization.
        if self.attn_pdrop is not None and self.training:
            attention_weights = F.dropout(attention_weights, p=self.attn_pdrop, training=self.training)

        # the query says: “What am I looking for?”
        # the keys say: “What kind of information does each token have?”
        # the attention weights decide: “Which tokens are relevant, and by how much?”
        # the values are the actual information to retrieve
        # the context vector is the final “retrieved information”
        context = torch.matmul(attention_weights, V)

        # Compute the output vector.
        # we need to reshape the context tensor to match the original shape of X.
        context = context.transpose(1, 2).contiguous().view(batch_size, seq_length, num_heads * d_k)
        output = self.out_linear(context)

        return attention_weights, output

# Example usage:
d_model = 512  # Embedding size
num_heads = 8  # Number of attention heads
d_k = 64  # Dimension per head (num_heads * d_k must be d_model)

batch_size = 2
seq_length = 10

# Random input tensor
X = torch.rand(batch_size, seq_length, d_model)

# Instantiate and apply Multi-Head Attention
multi_head_attn = MultiHeadAttention(d_model, num_heads, d_k, attn_pdrop=0.0)
_, C = multi_head_attn(X)

print(C.shape)  # Expected: (batch_size, seq_length, d_model)


torch.Size([2, 10, 512])


In [11]:

# ----------- Unit tests ---------------
# -----------DO NOT EDIT THIS PART -----


import unittest
import torch

class TestMultiHeadAttention(unittest.TestCase):
    def setUp(self):
        self.d_model = 512
        self.num_heads = 8
        self.d_k = 64
        self.batch_size = 2
        self.seq_length = 10
        torch.manual_seed(42)
        self.multi_head_attn = MultiHeadAttention(self.d_model, self.num_heads, self.d_k)
        self.X = torch.rand(self.batch_size, self.seq_length, self.d_model, requires_grad=True)

    def test_output_shape(self):
        _, output = self.multi_head_attn(self.X)
        expected_shape = (self.batch_size, self.seq_length, self.d_model)
        self.assertEqual(output.shape, expected_shape, f"Expected shape {expected_shape}, but got {output.shape}")

    def test_deterministic_behavior(self):
        torch.manual_seed(42)
        _, output1 = self.multi_head_attn(self.X)
        torch.manual_seed(42)
        _, output2 = self.multi_head_attn(self.X)
        self.assertTrue(torch.allclose(output1, output2, atol=1e-6), "MultiHeadAttention is not deterministic!")

    def test_gradient_computation(self):
        """Ensure gradients are computed properly."""
        _, output = self.multi_head_attn(self.X)
        loss = output.sum()  # Simple loss function
        loss.backward()

        self.assertIsNotNone(self.X.grad, "Gradients were not computed for input!")
        self.assertGreater(self.X.grad.abs().sum().item(), 0, "Gradient sum is zero!")

    def test_attention_softmax(self):
        """Ensure that the attention scores sum up to ~1."""
        attention_weights, _ = self.multi_head_attn(X)

        # Sum of softmax probabilities along last dimension should be close to 1
        attention_sum = attention_weights.sum(dim=-1)
        ones = torch.ones_like(attention_sum)

        self.assertTrue(torch.allclose(attention_sum, ones, atol=1e-6), "Attention weights do not sum to 1!")

    def test_known_computation(self):
        d_model = 4
        num_heads = 1
        d_k = 4
        seq_length = 3
        batch_size = 1
        multi_head_attn = MultiHeadAttention(d_model, num_heads, d_k)
        with torch.no_grad():
            # For Q, K, V projections
            for linear in [multi_head_attn.qkv_projection.q_linear,
                           multi_head_attn.qkv_projection.k_linear,
                           multi_head_attn.qkv_projection.v_linear,
                           multi_head_attn.out_linear]:
                linear.weight.copy_(torch.eye(d_model))
                if linear.bias is not None:
                    linear.bias.zero_()
        X_known = torch.ones(batch_size, seq_length, d_model)
        attention_weights, output = multi_head_attn(X_known)
        expected_output = torch.ones(batch_size, seq_length, d_model)
        self.assertTrue(torch.allclose(output, expected_output, atol=1e-6),
                        f"Expected output {expected_output}, but got {output}")

    def test_padding_mask(self):
        """Ensure that padding mask prevents attending to padded positions."""
        d_model = 4
        num_heads = 1
        d_k = 4
        seq_length = 4
        batch_size = 1

        multi_head_attn = MultiHeadAttention(d_model, num_heads, d_k)
        X_test = torch.rand(batch_size, seq_length, d_model)
        # Mask out last 2 positions (padding)
        attn_mask = torch.tensor([[1, 1, 0, 0]])
        attention_weights, _ = multi_head_attn(X_test, attention_mask=attn_mask)
        # Attention weights to padded positions should be ~0
        self.assertTrue(
            torch.allclose(attention_weights[:, :, :, 2:], torch.zeros_like(attention_weights[:, :, :, 2:]), atol=1e-6),
            "Attention weights to padded positions should be zero!"
        )

unittest.main(argv=[''], exit=False)

./Users/timli/miniconda3/lib/python3.12/site-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
.....
----------------------------------------------------------------------
Ran 6 tests in 0.216s

OK


Now let's implement the feed forward network and the layer normalization.

# Why do we do Layer Normalization?

Deep neural networks can suffer from unstable training dynamics, where the distribution of activations in intermediate layers shifts significantly during training (internal covariate shift). Normalization techniques help mitigate this. While Batch Normalization is common in computer vision, Layer Normalization is often preferred in Transformers and NLP.

Concept: Layer Normalization works by normalizing the features across the embedding dimension for each token independently within a layer. It calculates the mean and standard deviation for all features of a single token and uses them to normalize that token’s feature vector. Unlike Batch Norm, its calculations don’t depend on the batch size, making it more stable for variable sequence lengths common in NLP.

# What is the feed forward network?

After the attention mechanism processes the relationships between tokens, each position’s resulting vector is passed through a Position-wise Feed-Forward Network (FFN) independently. This provides additional non-linear processing capacity to the model.

Here we can just see that it is a two layer neural network with ReLU activation and dropout.

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super(FeedForward, self).__init__()

        self.fc1 = nn.Linear(d_model, d_ff)
        self.activation = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.fc2 = nn.Linear(d_ff, d_model)


    def forward(self, x: Tensor) -> Tensor:
        x = self.fc1(x)
        x = self.activation(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x

class LayerNorm(nn.Module):
    def __init__(self, hidden_size, eps: float):
        super(LayerNorm, self).__init__()
        self.gamma = nn.Parameter(torch.ones(hidden_size))
        self.beta = nn.Parameter(torch.zeros(hidden_size))
        self.eps = eps

    def forward(self, x: Tensor) -> Tensor:
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, unbiased=False, keepdim=True)
        normed = (x - mean) / torch.sqrt(var + self.eps)
        return normed * self.gamma + self.beta